# DEST — Rerun Collatz solo (200–209) para verificar pérdida

**STANDALONE — Ejecuta todo**

* **Por qué solo Collatz:** Halton/Sobol ya están verificados (45k únicos, sin duplicados, tau~0) y empataron con azar — no tienen bug. Collatz tiene **98% duplicados** (715 únicos/45k) y `42 vs 200 tau=0.18` — su `+0.76` en 42–61 fue suerte del offset.
* **Qué hace:** `CIFAR-10 → ResNet9`, 15 épocas, batch 128, `stochastic` vs `collatz_v3` (alpha coseno 0→0.5), **10 seeds 200–209** (20 runs, ~70 min T4), pareado por seed.
* **Para comparar:** si V3 vuelve a perder ~−0.69 (p≈0.003), confirma que 42–61 fue outlier. Si empata, fue varianza.


In [ ]:
# 0. Setup standalone
import os, sys, subprocess, shutil, glob
print("🔧 Setup...")
if os.path.exists("DEST"):
    subprocess.call(["rm","-rf","DEST"])
print("Clonando https://github.com/starlyn2010/DEST ...")
subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
subprocess.check_call([sys.executable,"-m","pip","install","-e","DEST","-q"])
if "DEST/src" not in sys.path: sys.path.insert(0, "DEST/src")
import dest
sys.modules["dest_lib"]=dest
for sub in ["config","samplers","models","datasets","runner","metrics","reproducibility"]:
    try:
        m=__import__(f"dest.{sub}", fromlist=[sub]); sys.modules[f"dest_lib.{sub}"]=m
    except: pass
print("✅ DEST instalado")
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Verificar duplicados rápido
from dest.samplers import CollatzFix3Sampler
class Dummy: 
    def __len__(self): return 45000
s=CollatzFix3Sampler(Dummy(), total_epochs=15, seed=42); s.set_epoch(0); vals=s._collatz_step
print("✅ CollatzFix3Sampler OK")


In [ ]:
# 1. Config — solo Collatz vs Stochastic
from dest_lib.config import get_config
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["stochastic","collatz_v3"]
config["seeds"]=list(range(200,210))
config["epochs"]=15
config["batch_size"]=128
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["output_dir"]="./dest_collatz_rerun"
config["val_fraction"]=0.1
config["verbose"]=True
import json
print(json.dumps({k:config[k] for k in ["datasets","samplers","seeds","epochs","output_dir"]},indent=2))
print(f"Total runs: {len(config['seeds'])*len(config['samplers'])} (20) — ~70 min T4")


In [ ]:
# 2. Ejecutar reanudable (par por seed)
import os, time, json
from dest_lib.runner import ExperimentRunner
runner=ExperimentRunner(config)
total=len(config["seeds"])*len(config["samplers"])
done=0
start_all=time.time()
for seed in config["seeds"]:
    for sampler_name in config["samplers"]:
        exp_id=f"CIFAR10_{sampler_name}"
        out_file=os.path.join(config["output_dir"], f"{exp_id}_{sampler_name}_seed_{seed}.json")
        if os.path.exists(out_file):
            try:
                j=json.load(open(out_file))
                if j.get("status")=="COMPLETE" and len(j.get("test_accs",[]))==15:
                    print(f"⏭️ Saltando {sampler_name} {seed} ({j['final_test_acc']:.2f}%)")
                    done+=1; continue
                else: os.remove(out_file)
            except: os.remove(out_file) if os.path.exists(out_file) else None
        print(f"\n[{done+1}/{total}] {sampler_name} seed {seed}")
        r=runner.run_single_seed(exp_id=exp_id, sampler_name=sampler_name, seed=seed, dataset="CIFAR10")
        print(f"✅ {sampler_name} {seed}: {r.final_test_acc:.2f}% en {r.total_runtime_seconds/60:.1f} min")
        done+=1
print(f"\n✅ Rerun completo {done}/{total} en {(time.time()-start_all)/60:.1f} min")


In [ ]:
# 3. Resumen pareado
import glob, json, numpy as np
from collections import defaultdict
files=[f for f in glob.glob("dest_collatz_rerun/*.json") if "sampler_name" in json.load(open(f))]
print(f"JSONs válidos: {len(files)}/20")
if files:
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j["sampler_name"]].append(j)
    for s in ["stochastic","collatz_v3"]:
        arr=[j["final_test_acc"] for j in groups[s]]
        if arr: print(f"{s:12s}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f} n={len(arr)}")
    if "stochastic" in groups and "collatz_v3" in groups:
        stoch={j["seed"]:j["final_test_acc"] for j in groups["stochastic"]}
        v3={j["seed"]:j["final_test_acc"] for j in groups["collatz_v3"]}
        common=sorted(set(stoch)&set(v3))
        diffs=[v3[s]-stoch[s] for s in common]
        if diffs:
            from scipy import stats
            t,p=stats.ttest_rel([v3[s] for s in common],[stoch[s] for s in common])
            d=np.mean(diffs)/np.std(diffs,ddof=1)
            print(f"\nV3 vs stoch: diff {np.mean(diffs):+.3f} p={p:.4f} d={d:.2f} gana {sum(d>0 for d in diffs)}/{len(diffs)}")
            print(f"95% CI [{np.mean(diffs)-1.96*np.std(diffs,ddof=1)/np.sqrt(len(diffs)):.3f}, {np.mean(diffs)+1.96*np.std(diffs,ddof=1)/np.sqrt(len(diffs)):.3f}]")
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    for s in ["stochastic","collatz_v3"]:
        if s not in groups: continue
        arr=np.array([j["test_accs"] for j in groups[s]])
        plt.plot(range(1,16), arr.mean(0), label=s)
        plt.fill_between(range(1,16), arr.mean(0)-arr.std(0,ddof=1), arr.mean(0)+arr.std(0,ddof=1), alpha=0.15)
    plt.xlabel("Época"); plt.ylabel("Test acc %"); plt.title("CIFAR-10 Rerun Collatz vs Stoch (200-209)")
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig("dest_collatz_rerun/curvas.png", dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# 4. Zip y descarga
import shutil, os, glob, json
files=[f for f in glob.glob("dest_collatz_rerun/*.json") if "sampler_name" in json.load(open(f))]
print(f"JSONs válidos: {len(files)}/20")
if files:
    shutil.make_archive("resultados_Collatz_Rerun_200_209","zip","dest_collatz_rerun")
    print(f"✅ ZIP {os.path.getsize('resultados_Collatz_Rerun_200_209.zip')/1e6:.2f} MB")
    try:
        from google.colab import files; files.download("resultados_Collatz_Rerun_200_209.zip")
    except: print(os.path.abspath("resultados_Collatz_Rerun_200_209.zip"))


**Al terminar:** sube el ZIP a `Redes liquidas/dest/dest_results_paper/` y compara con `42–61` (`+0.76` vs `-0.69`). Si vuelve a perder, confirma que `42–61` fue outlier por offset.
